In [ ]:
import json
import logging
import os
import sys
import time
from pathlib import Path

for candidate in ("../src", "./src"):
    resolved = os.path.abspath(candidate)
    if resolved not in sys.path:
        sys.path.insert(0, resolved)

from dbx_metadata_generator.review_csv import (
    COLUMN_DESCRIPTION,
    TABLE_DESCRIPTION,
    parse_write_flag,
    read_review_csv,
)


DEFAULTS = {
    "csv_path": "",
    "allow_stale_metadata": "false",
    "log_level": "INFO",
}


def _widget(name: str, default: str, label: str) -> str:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)
    return dbutils.widgets.get(name).strip()


def _as_bool(value: str) -> bool:
    return value.strip().casefold() in {"true", "1", "yes", "y"}


CSV_PATH_SETTING = _widget("csv_path", DEFAULTS["csv_path"], "Edited metadata CSV path")
if not CSV_PATH_SETTING:
    raise ValueError("csv_path is required")
CSV_PATH = Path(CSV_PATH_SETTING).expanduser()
ALLOW_STALE_METADATA = _as_bool(_widget("allow_stale_metadata", DEFAULTS["allow_stale_metadata"], "Apply when current metadata changed since export"))
LOG_LEVEL_NAME = _widget("log_level", DEFAULTS["log_level"], "Progress log level").upper()
LOG_LEVEL = getattr(logging, LOG_LEVEL_NAME, None)
if not isinstance(LOG_LEVEL, int):
    raise ValueError(f"Unsupported log level: {LOG_LEVEL_NAME}")

RUN_STARTED_AT = time.monotonic()
LOG = logging.getLogger("dbx_metadata_csv_apply")
LOG.setLevel(LOG_LEVEL)
LOG.propagate = False
if not any(getattr(handler, "_dbx_metadata_csv_apply", False) for handler in LOG.handlers):
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(message)s"))
    handler._dbx_metadata_csv_apply = True
    LOG.addHandler(handler)


def progress(stage: str, message: str, *, level: int = logging.INFO, **details) -> None:
    payload = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "elapsed_seconds": round(time.monotonic() - RUN_STARTED_AT, 1),
        "stage": stage,
        "message": message,
        **details,
    }
    LOG.log(level, json.dumps(payload, default=str, sort_keys=True))

REVIEW_ROWS = read_review_csv(CSV_PATH)
SELECTED_ROWS = [row for row in REVIEW_ROWS if parse_write_flag(row["proposed_write"])]
progress(
    "configuration",
    "Edited metadata CSV loaded",
    csv_path=str(CSV_PATH),
    review_rows=len(REVIEW_ROWS),
    selected_rows=len(SELECTED_ROWS),
    allow_stale_metadata=ALLOW_STALE_METADATA,
)


# Apply Metadata Review CSV

Set `csv_path` to an edited CSV produced by `semantic_metadata_generator`. Rows are applied only
when `proposed_write` is true. A blank `new_metadata` clears the table or column description.

Before issuing any DDL, the notebook validates every selected target and checks that its live comment
still matches `current_metadata`. Set `allow_stale_metadata=true` only when overwriting a newer comment
is intentional.


In [ ]:
def quote_identifier(value: str) -> str:
    return f"`{value.replace('`', '``')}`"


def target_identifier(row: dict[str, str]) -> str:
    parts = [row["catalog_name"], row["schema_name"], row["table_name"]]
    if row["target_type"].strip().upper() == COLUMN_DESCRIPTION:
        parts.append(row["column_name"])
    return ".".join(quote_identifier(part) for part in parts)


def comment_sql_value(value: str) -> str:
    if value == "":
        return "NULL"
    return "'" + value.replace("'", "''") + "'"


def current_metadata(row: dict[str, str]) -> str:
    args = {
        "catalog": row["catalog_name"],
        "schema": row["schema_name"],
        "table": row["table_name"],
    }
    if row["target_type"].strip().upper() == TABLE_DESCRIPTION:
        result = spark.sql(
            """
            SELECT comment
            FROM system.information_schema.tables
            WHERE table_catalog = :catalog AND table_schema = :schema AND table_name = :table
            """,
            args=args,
        ).first()
    else:
        args["column"] = row["column_name"]
        result = spark.sql(
            """
            SELECT comment
            FROM system.information_schema.columns
            WHERE table_catalog = :catalog AND table_schema = :schema
              AND table_name = :table AND column_name = :column
            """,
            args=args,
        ).first()
    if result is None:
        raise ValueError(f"Metadata target no longer exists or is not visible: {row['metadata_path']}")
    return result["comment"] or ""


def comment_ddl(row: dict[str, str]) -> str:
    target_type = row["target_type"].strip().upper()
    target = target_identifier(row)
    value = comment_sql_value(row["new_metadata"])
    if target_type == TABLE_DESCRIPTION:
        return f"COMMENT ON TABLE {target} IS {value}"
    return f"COMMENT ON COLUMN {target} IS {value}"


In [ ]:
progress("preflight", "Checking selected metadata targets", selected_rows=len(SELECTED_ROWS))
PREFLIGHT_ROWS = []
STALE_ROWS = []
for row in SELECTED_ROWS:
    live_metadata = current_metadata(row)
    preflight = {**row, "live_metadata": live_metadata}
    PREFLIGHT_ROWS.append(preflight)
    if live_metadata != row["current_metadata"]:
        STALE_ROWS.append({
            "metadata_path": row["metadata_path"],
            "csv_current_metadata": row["current_metadata"],
            "live_metadata": live_metadata,
        })

if STALE_ROWS and not ALLOW_STALE_METADATA:
    display(spark.createDataFrame(STALE_ROWS))
    raise RuntimeError(
        f"Preflight stopped: {len(STALE_ROWS)} selected target(s) changed since the CSV was exported"
    )

progress(
    "apply",
    "Applying selected metadata changes",
    selected_rows=len(PREFLIGHT_ROWS),
    stale_overrides=len(STALE_ROWS),
)
APPLY_RESULTS = []
for index, row in enumerate(PREFLIGHT_ROWS, start=1):
    progress(
        "apply",
        "Applying metadata change",
        current=index,
        total=len(PREFLIGHT_ROWS),
        metadata_path=row["metadata_path"],
    )
    try:
        spark.sql(comment_ddl(row))
        verified_metadata = current_metadata(row)
        if verified_metadata != row["new_metadata"]:
            raise RuntimeError("Metadata verification did not match the requested value")
        APPLY_RESULTS.append({
            "metadata_path": row["metadata_path"],
            "status": "applied",
            "new_metadata": row["new_metadata"],
            "error": "",
        })
    except Exception as exc:
        APPLY_RESULTS.append({
            "metadata_path": row["metadata_path"],
            "status": "error",
            "new_metadata": row["new_metadata"],
            "error": f"{type(exc).__name__}: {str(exc)[:1000]}",
        })
        progress(
            "apply",
            "Metadata change failed",
            level=logging.ERROR,
            metadata_path=row["metadata_path"],
            error_type=type(exc).__name__,
            error=str(exc)[:300],
        )

if APPLY_RESULTS:
    display(spark.createDataFrame(APPLY_RESULTS).orderBy("metadata_path"))

ERROR_RESULTS = [result for result in APPLY_RESULTS if result["status"] == "error"]
RUN_SUMMARY = {
    "csv_path": str(CSV_PATH),
    "review_row_count": len(REVIEW_ROWS),
    "selected_row_count": len(SELECTED_ROWS),
    "stale_override_count": len(STALE_ROWS),
    "applied_count": len(APPLY_RESULTS) - len(ERROR_RESULTS),
    "error_count": len(ERROR_RESULTS),
}
progress("complete", "Metadata CSV application finished", **RUN_SUMMARY)
print(json.dumps(RUN_SUMMARY, indent=2))
if ERROR_RESULTS:
    raise RuntimeError(json.dumps(RUN_SUMMARY))
dbutils.notebook.exit(json.dumps(RUN_SUMMARY))
